# SSTW paper evidence postprocess Colab 入口

该 Notebook 负责恢复 runtime、motion threshold 和 formal comparison scoring 阶段包, 生成最终门禁前所需的论文辅助证据: motion consistency 处理、内部消融、adaptive attack、replay/sketch 或 Claim-3 downgrade、统计置信区间、低 FPR 阻断记录和数据切分泄漏检查。

Notebook 只调用仓库 helper, 不直接手写 records、tables、figures 或 reports。


In [ ]:
# 1. 挂载 Google Drive 并检查 GPU
from google.colab import drive
drive.mount('/content/drive')

import json
import os
import subprocess
import sys
from pathlib import Path

print(subprocess.getoutput('nvidia-smi'))
DRIVE_PROJECT_ROOT = os.environ.get('SSTW_DRIVE_PROJECT_ROOT', '/content/drive/MyDrive/SSTW')
Path(DRIVE_PROJECT_ROOT).mkdir(parents=True, exist_ok=True)
print('drive_project_root =', DRIVE_PROJECT_ROOT)


In [ ]:
# 1.1 可编辑 workflow profile 切换
# 修改 SSTW_WORKFLOW_PROFILE_VALUE 即可切换当前 Notebook 的运行层级。
# 留空表示使用当前 Notebook role 在统一配置中的默认 profile。
# 可选值: 'validation_scale', 'pilot_paper', 'full_paper'。
# 典型用法: 当前 Notebook 使用 'validation_scale'、'pilot_paper' 或未来启用后的 'full_paper'。
import os

SSTW_WORKFLOW_PROFILE_VALUE = 'validation_scale'  # 当前显式运行 validation-scale

if SSTW_WORKFLOW_PROFILE_VALUE.strip():
    os.environ['SSTW_WORKFLOW_PROFILE'] = SSTW_WORKFLOW_PROFILE_VALUE.strip()
    print('SSTW_WORKFLOW_PROFILE =', os.environ['SSTW_WORKFLOW_PROFILE'])
else:
    os.environ.pop('SSTW_WORKFLOW_PROFILE', None)
    print('SSTW_WORKFLOW_PROFILE 未显式设置, 将使用当前 Notebook role 默认值。')


In [ ]:
# 2. 冷启动获取仓库代码
# Colab 环境不会保留代码, 每次运行都需要 clone 或 pull。
REPO_URL = os.environ.get('SSTW_REPO_URL', 'https://github.com/RICHAAARC/SSTW.git')
REPO_DIR = os.environ.get('SSTW_REPO_DIR', '/content/SSTW')

if not Path(REPO_DIR).exists():
    if not REPO_URL:
        raise RuntimeError('请先设置 SSTW_REPO_URL, 或将仓库上传到 /content/SSTW')
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    print('仓库目录已存在, 执行 git pull 以同步远端代码。')
    %cd "$REPO_DIR"
    !git pull --ff-only
%cd "$REPO_DIR"
!git rev-parse --short HEAD


In [ ]:
# 3. 安装 Colab GPU 运行依赖
%pip install -U git+https://github.com/huggingface/diffusers transformers accelerate safetensors imageio imageio-ffmpeg sentencepiece protobuf huggingface_hub


In [ ]:
# 4. 可选 Hugging Face 认证
# token 只写入当前 Colab 环境变量, 不写入 records、tables、reports 或 package manifest。
try:
    from huggingface_hub import login
except Exception as exc:
    raise RuntimeError('huggingface_hub 未安装或无法导入, 请重新执行依赖安装单元') from exc

if os.environ.get('HF_TOKEN'):
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
    print('HF_TOKEN 状态: provided from environment.')
else:
    print('HF_TOKEN 状态: not_provided in environment, 将尝试匿名下载公开模型。')


In [ ]:
# 5. 读取统一 workflow profile 配置, 初始化 Drive 归档布局, 并切换本地阶段 workspace
from paper_workflow.notebook_utils import generative_video_model_probe_workflow as probe_workflow
from paper_workflow.colab_utils.stage_package_sync import prepare_colab_stage_layout

# 本开关使后续 runner 读写 /content 本地 workspace, 阶段完成后再以 zip 形式回写 Drive。
os.environ.setdefault('SSTW_COLAB_STAGE_IO_MODE', 'local_zip')

NOTEBOOK_ROLE = 'paper_evidence_postprocess'
WORKFLOW_PROFILE = os.environ.get('SSTW_WORKFLOW_PROFILE') or probe_workflow.default_workflow_profile_for_notebook_role(NOTEBOOK_ROLE)
workflow_profile = probe_workflow.resolve_notebook_workflow_profile(WORKFLOW_PROFILE, NOTEBOOK_ROLE)
PROFILE = workflow_profile['runtime_profile']
drive_layout = probe_workflow.ensure_drive_layout(
    DRIVE_PROJECT_ROOT,
    workflow_profile=WORKFLOW_PROFILE,
    notebook_role=NOTEBOOK_ROLE,
)
layout = prepare_colab_stage_layout(
    drive_layout,
    notebook_role=NOTEBOOK_ROLE,
)

print(json.dumps({
    'workflow_profile': workflow_profile,
    'drive_layout': drive_layout,
    'active_local_layout': layout,
    'enabled_stage_plan': probe_workflow.build_workflow_stage_plan(WORKFLOW_PROFILE, NOTEBOOK_ROLE),
}, ensure_ascii=False, indent=2))


In [ ]:
# 6. 执行统一配置的 paper evidence postprocess stage plan
# Notebook 只切换 workflow profile, 不维护阶段顺序、命令映射或产物清单。
INCLUDE_VIDEOS_IN_PACKAGE = os.environ.get('SSTW_INCLUDE_VIDEOS_IN_PACKAGE', 'false').lower() == 'true'

paper_evidence_result = probe_workflow.run_configured_colab_stage_plan(
    layout,
    workflow_profile=WORKFLOW_PROFILE,
    notebook_role=NOTEBOOK_ROLE,
    include_videos=INCLUDE_VIDEOS_IN_PACKAGE,
)
print(json.dumps(paper_evidence_result, ensure_ascii=False, indent=2))
